# Генерация кандидатов

## Цель этапа

На предыдущем этапе были построены два baseline:

- `Popular` — глобально популярные товары;
- `Recent Items + Popularity fallback` — последние товары из истории текущей сессии.

`Recent Items` значительно улучшил качество рекомендаций, однако у такого подхода есть существенный минус - он работает только на товары, которые встречались у пользователя в сессии.

Цель этого этапа — сформировать для каждой сессии расширенный набор потенциально
релевантных товаров, а именно множество **`candidate set`**.

Для генераци будем использовать несколько источников кандидатов:
1. товары из истории текущей сессии;
2. товары, которые часто встречаются вместе в других сессиях (`co-visitation`);
3. популярные товары как fallback.

   
Качество генерации кандидатов будем оценивать с помощью `Recall@K` отдельно для `clicks`, `carts` и `orders`. На следующем этапе полученные кандидаты будут ранжироваться ML-моделью.

In [4]:
import json 
import pandas as pd
from pathlib import Path

data_path = Path("../data/otto-recsys-train.jsonl")

df = pd.read_json(
    data_path,
    lines=True,
    nrows=10000
)

## Подготовка данных

На этапе разработки используется выборка из 10 000 сессий.

Как и на этапе валидации, каждую сессию разделим хронологически:

- первые 75% событий — `history`;
- последние 25% событий — `future`.

'history' используется для построения рекомендательной системы и генерации
кандидатов, а 'future' — только для оценки качества.

Такое разделение имитирует реальную ситуацию: система видит прошлые действия
пользователя и пытается предсказать его последующие взаимодействия.

In [5]:
history_arr = []
future_arr = []

for events in df['events']:
    future_size = max(1, int(len(events) * 0.25))

    history = events[:-future_size]
    future = events[-future_size:]

    history_arr.append(history)
    future_arr.append(future)

df['history'] = history_arr
df['future'] = future_arr

## Co-visitation

Для поиска новых потенциально релевантных товаров будем использовать совместную встречаемость товаров в историях сессий.

Если два товара встречаются в одной сессии, их связь получает вес `1` независимо от количества повторных взаимодействий с этими товарами внутри данной сессии.

Например:

`A → A → A → B`

увеличивает связь между `A` и `B` на `1`, а не на `3`.

Мы будем придерживаться такой логики, потому что строим связь именно A → B, а частотность A указывает именно на заинтересованность пользователя к товару A. Также проблему такого подхода может дать пользователь, который искусственно раздует связь между товарами. Например, если он 15 раз открыл товар C, а затем 1 раз открыл товар D, то это еще не означает, что связь C → D должна быть в 50 раз сильнее.

### Построение

Для каждой сессии возьмем уникальные товары из 'history' и далее рассмотрим пары этих товаров.

Для каждой пары товаров будем считать количество сессий, когда они встретились вместе.

Используем только сессии из 'history', чтобы избежать leekage.

In [6]:
from itertools import combinations

pair_counts = {}

for session in df['history']:
    unique_items = set()

    for event in session:
        unique_items.add(event['aid'])

    for item_1, item_2 in combinations(unique_items, 2):
        pair = (item_1, item_2)
        if pair not in pair_counts:
            pair_counts[pair] = 0
        pair_counts[pair] += 1

print('Количество связей:', len(pair_counts))

Количество связей: 10037784


### Обновленный список связей товаров

Чтобы не хранить все связи между товарами, стоит найти силу их связи, равную количеству сессий, в которых они встречаются вместе. Самые сильные связи отберем для дальнейшего построения рекомендательной системы.

In [8]:
item_neighbors = {}

for (item_1, item_2), count in pair_counts.items():
    if item_1 not in item_neighbors:
        item_neighbors[item_1] = []
    if item_2 not in item_neighbors:
        item_neighbors[item_2] = [] 

    item_neighbors[item_1].append((item_2, count))
    item_neighbors[item_2].append((item_1, count))

Для каждого товара найдем 20 соседей с наибольшей силой связи.

In [10]:
def get_count(pair):
    return pair[1]

top_n = 20

for item in item_neighbors:
    item_neighbors[item] = sorted(
        item_neighbors[item],
        key=get_count,
        reverse=True
    )[:top_n]

Теперь проверим структуру на одном из товаров.

In [16]:
import random

testing_item = random.choice(list(item_neighbors.keys()))

print('Товар:', testing_item)
print('Количество соседей:', len(item_neighbors[testing_item]))

for neighbor, count in item_neighbors[testing_item]:
    print('Сосед:', neighbor, '| Сила связи:', count)

Товар: 657256
Количество соседей: 20
Сосед: 1725190 | Сила связи: 5
Сосед: 409671 | Сила связи: 4
Сосед: 29735 | Сила связи: 4
Сосед: 328087 | Сила связи: 4
Сосед: 274873 | Сила связи: 4
Сосед: 1487164 | Сила связи: 4
Сосед: 1851771 | Сила связи: 4
Сосед: 289346 | Сила связи: 3
Сосед: 328678 | Сила связи: 3
Сосед: 832192 | Сила связи: 3
Сосед: 6851 | Сила связи: 3
Сосед: 1208474 | Сила связи: 3
Сосед: 274873 | Сила связи: 3
Сосед: 1261740 | Сила связи: 3
Сосед: 923973 | Сила связи: 3
Сосед: 332654 | Сила связи: 3
Сосед: 1065670 | Сила связи: 2
Сосед: 354638 | Сила связи: 2
Сосед: 1708468 | Сила связи: 2
Сосед: 1036471 | Сила связи: 2


### Генерация кандидатов из co-visitation

Теперь настало время для каждого товара из текущей сессии найти соседей с наибольшей связью.

Соседи всех товаров будут объеденены в общий набора кандидатов. Если один и тот же кандидат с несколькими товарами, то сила его связи будет суммироваться. То есть товары, которые связаны сразу с несколькими товарами из текущей сессии, будут иметь больший итоговый score по co-visitation.

`Пример:`
history: A, F

A → X: 50
A → Y: 30

F → X: 70
F → Z: 40

X → 50 + 70 = 120
Y → 30
Z → 40

Самым предпочтительным кандидатом для выбора будет X.

In [22]:
def generation_covis_candidate(history, item_neighbors, k=100):
    candidate_scores = {}

    # Здесь хранятся уникальные товары из 'history' сессии
    history_items = set()

    for event in history:
        history_items.add(event['aid'])

    # co-visitation для каждого товара
    for item in history_items:
        if item not in item_neighbors:
            continue

        for neighbor, count in item_neighbors[item]:
            if neighbor not in candidate_scores:
                candidate_scores[neighbor] = 0
            candidate_scores[neighbor] += count
            
    candidates = []
    for candidate in candidate_scores.items():
        candidates.append(candidate)
    candidates = sorted(candidates, key=get_count, reverse=True)

    res = []
    for item, score in candidates[:k]:
        res.append(item)

    return res

### Проверка работы генерации

Для проверки работы генерации возьмем одну сессий.

In [23]:
test_history = df['history'].iloc[0]

test_candidates = generation_covis_candidate(
    test_history,
    item_neighbors,
    k=100
) 

print('Количество кандидатов:', len(test_candidates))
print('Первые 20 кандидатов:')
print(test_candidates[:20])

Количество кандидатов: 100
Первые 20 кандидатов:
[1813509, 1460239, 832192, 1750538, 724999, 1446430, 1767945, 102416, 218130, 1110548, 72748, 984597, 10268, 1517085, 1582117, 946219, 97836, 504365, 1171505, 29735]


## Оценка качества генерации кандидатов

Проверим качество генерации, используя метрику **Recall@100**.

In [24]:
clicks_arr = []
carts_arr = []
orders_arr = []

for future in df['future']:
    clicks = set()
    carts = set()
    orders = set()

    for event in future:
        if event['type'] == 'clicks':
            clicks.add(event['aid'])

        elif event['type'] == 'carts':
            carts.add(event['aid'])

        elif event['type'] == 'orders':
            orders.add(event['aid'])

    clicks_arr.append(clicks)
    carts_arr.append(carts)
    orders_arr.append(orders)

df['future_clicks'] = clicks_arr
df['future_carts'] = carts_arr
df['future_orders'] = orders_arr

In [25]:
def evaluate_candidate_recall(histories, targets, item_neighbors, k=100):
    recalls = []

    for history, relevant_items in zip(histories, targets):
        if len(relevant_items) == 0:
            continue
            
        candidates = generation_covis_candidate(history, item_neighbors,k=k)
        hits = 0
        for item in candidates:
            if item in relevant_items:
                hits += 1

        recall = hits / len(relevant_items)
        recalls.append(recall)

    return sum(recalls) / len(recalls)

In [26]:
recall_clicks = evaluate_candidate_recall(
    df['history'],
    df['future_clicks'],
    item_neighbors,
    k=100
)

recall_carts = evaluate_candidate_recall(
    df['history'],
    df['future_carts'],
    item_neighbors,
    k=100
)

recall_orders = evaluate_candidate_recall(
    df['history'],
    df['future_orders'],
    item_neighbors,
    k=100
)


print('Clicks Recall@100:', recall_clicks)
print('Carts Recall@100:', recall_carts)
print('Orders Recall@100:', recall_orders)

Clicks Recall@100: 0.21814682411734584
Carts Recall@100: 0.2717631372654386
Orders Recall@100: 0.399402998651656


### Результаты генерации co-visitation

При генерации 100 кандидатов среди товаров получены следующие значения:

- `Clicks Recall@100` ≈ 0.218;
- `Carts Recall@100` ≈ 0.272;
- `Orders Recall@100` ≈ 0.399.

Текущие метрики хуже нашего baseline, потому что генератор пока не включает товары из 'history'. А как мы поняли из прошлых тестов и исследований, повторные взаимодействия с товарами встречаются доавольно часто и это стоит учитывать.

## Объединение history и co-visitation

Поняв проблему, мы объединим два источника кандидатов:

- товары из 'history', начиная с наиболее недавних;
- кандидаты, полученные с помощью co-visitation.

In [28]:
def generation_combined_candidate(history, item_neighbors, k=100):
    candidates = []
    seen = set()

    # Кандидаты из history
    for event in reversed(history):
        aid = event['aid']

        if aid not in seen:
            candidates.append(aid)
            seen.add(aid)

        if len(candidates) == k:
            return candidates

    covis_candidates = generation_covis_candidate(
        history,
        item_neighbors,
        k=k
    )

    # Дополняем список
    for aid in covis_candidates:
        if aid not in seen:
            candidates.append(aid)
            seen.add(aid)

        if len(candidates) == k:
            break

    return candidates

Настал момент оценить качество нового способа генерации.

In [30]:
def ev_combined_recall(histories, targets, item_neighbors, k=100):
    recalls = []
    
    for history, relevant_items in zip(histories, targets):
        if len(relevant_items) == 0:
            continue

        candidates = generation_combined_candidate(history, item_neighbors,k=k)
        hits = 0
        for item in candidates:
            if item in relevant_items:
                hits += 1

        recall = hits / len(relevant_items)
        recalls.append(recall)

    return sum(recalls) / len(recalls)

In [32]:
combined_recall_clicks = ev_combined_recall(
    df['history'],
    df['future_clicks'],
    item_neighbors,
    k=100
)

combined_recall_carts = ev_combined_recall(
    df['history'],
    df['future_carts'],
    item_neighbors,
    k=100
)

combined_recall_orders = ev_combined_recall(
    df['history'],
    df['future_orders'],
    item_neighbors,
    k=100
)

print('Clicks Recall@100:', combined_recall_clicks)
print('Carts Recall@100:', combined_recall_carts)
print('Orders Recall@100:', combined_recall_orders)

Clicks Recall@100: 0.2806525293566961
Carts Recall@100: 0.35961147016030315
Orders Recall@100: 0.570419276758261


## Результаты метрики после объединения 

После объединения товаров из 'history' и кандидатов из co-visitation получены следующие результаты:

- **Clicks Recall@100 ≈ 0.281** (было ≈0.218)
- **Carts Recall@100 ≈ 0.360** (было ≈0.272)
- **Orders Recall@100 ≈ 0.570** (было ≈0.399)

Из сравнения метрик делаем вывод, что, используя объединенный метод генерации, нам удалось повысить качество для всех трех типов событий.

## Выбор количества кандидатов

Проверим качество объединенного метода генерации для различного количества кандидатов: `50`, `100`, `200`.

In [33]:
k_candidates = [50, 100, 200]

results = []

for k in k_candidates:
    recall_clicks = ev_combined_recall(
        df['history'],
        df['future_clicks'],
        item_neighbors,
        k=k
    )

    recall_carts = ev_combined_recall(
        df['history'],
        df['future_carts'],
        item_neighbors,
        k=k
    )

    recall_orders = ev_combined_recall(
        df['history'],
        df['future_orders'],
        item_neighbors,
        k=k
    )

    results.append([
        k,
        recall_clicks,
        recall_carts,
        recall_orders
    ])

results

[[50, 0.26609423258575043, 0.3320373816178592, 0.5429288912498434],
 [100, 0.2806525293566961, 0.35961147016030315, 0.570419276758261],
 [200, 0.28950601580014274, 0.37320899127445156, 0.5806300916486472]]

## Выбор размера пула кандидатов

Прослеживается тенденция, что с увеличением размера пула кандидатов растет и **Recall**. Однако наибольший скачок наблюдается при переходе от 50 к 100, нежели от 100 к 200. Поэтому в качестве основного размер пула выбрано `100` кандидатов на сессию, как некий компромисс между **Recall** и вычислительной нагрузкой.

Итоговый генератор будет объединять два источника: товары из 'history' и связанные товары из текущей сессии, найденные с помощью `co-visitation`. Размер пула кандидатов: `100`.